# Домашняя работа 2 — качество данных

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 2.2.3


## Независимая загрузка

In [2]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string[python]
activity_year    string[python]
lei              string[python]
county_code      string[python]
action_taken     string[python]
loan_amount      string[python]
income           string[python]
dtype: object


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,BAD
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## H1. Собственная функция очистки

In [3]:
def clean_export(source):

    # Копия входа, чтобы не менять исходный объект
    df = source.copy(deep=True)
    original_cols = list(source.columns)

    # Журнал: input, replays, clean, quarantine, parse_errors
    audit = {
        'input': len(df),
        'replays': 0,
        'clean': 0,
        'quarantine': 0,
        'parse_errors': 0,
    }

    # Технический ключ для поиска одинаковых source_row
    df['_row_key'] = pd.to_numeric(df['source_row'], errors='coerce')

    # Проверка конфликтующих версий известного source_row:
    # если у одного ключа остались разные строки после drop_duplicates — AssertionError
    drop_idx = []
    known = df['_row_key'].notna()
    for key, group in df.loc[known].groupby('_row_key', sort=False):
        if len(group) > 1:
            unique = group.drop_duplicates(subset=original_cols)
            assert len(unique) == 1, f'Конфликт версий для source_row {key}'
            # Исключение одинаковой повторной доставки: оставляем первую, остальные — replays
            drop_idx.extend(group.index[1:])

    dedup = df.drop(index=drop_idx).drop(columns=['_row_key'])
    audit['replays'] = len(drop_idx)

    # Сырые значения нужны для карантина и для подсчёта ошибок формата
    row_raw = dedup['source_row']
    action_raw = dedup['action_taken']
    loan_raw = dedup['loan_amount']
    income_raw = dedup['income']
    county_raw = dedup['county_code']

    # Числовые преобразования
    row_num = pd.to_numeric(row_raw, errors='coerce')
    action_num = pd.to_numeric(action_raw, errors='coerce')
    loan_num = pd.to_numeric(loan_raw, errors='coerce')
    income_num = pd.to_numeric(income_raw, errors='coerce')

    # Внешние пробелы территории убираем через strip
    county_norm = county_raw.str.strip()

    # Подсчёт новых ошибок преобразования: значение было, но числом не стало
    parse_errors = 0
    for raw, num in ((row_raw, row_num), (action_raw, action_num),
                     (loan_raw, loan_num), (income_raw, income_num)):
        parse_errors += int((raw.notna() & num.isna()).sum())
    audit['parse_errors'] = parse_errors

    # Маска обязательных правил и причина для карантина
    reasons = []
    for idx in dedup.index:
        r = []
        # Известный source_row
        if pd.isna(row_num.loc[idx]):
            r.append('некорректный source_row')
        # Исходы 1–3
        if pd.isna(action_num.loc[idx]) or action_num.loc[idx] not in (1, 2, 3):
            r.append('некорректный action_taken')
        # Известная положительная сумма
        if pd.isna(loan_num.loc[idx]) or loan_num.loc[idx] <= 0:
            r.append('некорректная сумма кредита')
        # Территория 11001
        county = county_norm.loc[idx]
        if pd.isna(county) or county != '11001':
            r.append('некорректная территория')
        # income может отсутствовать, но если значение есть — оно должно читаться числом
        if pd.notna(income_raw.loc[idx]) and pd.isna(income_num.loc[idx]):
            r.append('некорректный доход')
        reasons.append(';'.join(r))

    reasons = pd.Series(reasons, index=dedup.index, dtype='object')
    clean_mask = reasons == ''

    # Подготовленный слой с числовыми типами
    clean = dedup.loc[clean_mask, original_cols].copy()
    clean['source_row'] = row_num[clean_mask].astype('Int64')
    clean['action_taken'] = action_num[clean_mask].astype('Int64')
    clean['loan_amount'] = loan_num[clean_mask].astype(float)
    clean['income'] = income_num[clean_mask].astype(float)
    clean['county_code'] = county_norm[clean_mask]

    # Карантин с исходными значениями и reason
    quarantine = dedup.loc[~clean_mask, original_cols].copy()
    quarantine['reason'] = reasons[~clean_mask]

    audit['clean'] = len(clean)
    audit['quarantine'] = len(quarantine)

    # Баланс: вход = повторы + чистые + карантин
    assert audit['input'] == audit['replays'] + audit['clean'] + audit['quarantine']

    return clean, quarantine, audit

## H2. Год и известный доход

In [4]:
clean, quarantine, audit = clean_export(export)
print(f'Журнал: {audit}')

# Проверяем, что во всех подготовленных строках activity_year — "2023"
year_ok = clean['activity_year'].eq('2023').fillna(False).all()
print(f'Год везде 2023: {year_ok}')

# Общее количество, известный и неизвестный доход
n_total = len(clean)
known_income = int(clean['income'].notna().sum())
unknown_income = int(clean['income'].isna().sum())

# Средний известный доход и его единица (тысячи USD)
mean_income = clean['income'].mean()

print(f'Всего записей: {n_total}')
print(f'Известный доход: {known_income}')
print(f'Неизвестный доход: {unknown_income}')
print(f'Средний доход: {mean_income} тысяч USD')
print('Единица измерения: тысячи USD')

# Проверка баланса и сохранности входа
balance_ok = audit['input'] == audit['replays'] + audit['clean'] + audit['quarantine']
print(f'Баланс сходится: {balance_ok}')

assert year_ok
assert balance_ok
assert_frame_equal(export, raw_copy)
print('Вход не изменился')

Журнал: {'input': 6832, 'replays': 2, 'clean': 6827, 'quarantine': 3, 'parse_errors': 2}
Год везде 2023: True
Всего записей: 6827
Известный доход: 6485
Неизвестный доход: 342
Средний доход: 237.06553585196608 тысяч USD
Единица измерения: тысячи USD
Баланс сходится: True
Вход не изменился


## H3. Проверка отчётного года

In [5]:
# Создаём копию export и меняем год первой строки на "2022"
wrong_year = export.copy(deep=True)
wrong_year.loc[wrong_year.index[0], 'activity_year'] = '2022'

# Проверяем через eq("2023").fillna(False).all(), что нарушение обнаружено
year_ok_wrong = wrong_year['activity_year'].eq('2023').fillna(False).all()
print(f'Проверка года на испорченных данных: {year_ok_wrong}')
assert not year_ok_wrong

# Проверяем, что export сохранился
assert_frame_equal(export, raw_copy)
print('export сохранён')

# Пример исходного пропуска: известный пропуск income, строка осталась в clean
print('Примеры известных пропусков дохода:')
print(clean.loc[
    clean['income'].isna(),
    ['source_row', 'loan_amount', 'income']
].head())

# Пример новой ошибки преобразования: income было, но числом не стало
print('Примеры ошибок формата дохода:')
print(quarantine.loc[
    quarantine['reason'].str.contains('некорректный доход', na=False),
    ['source_row', 'loan_amount', 'income', 'reason']
].head())

# Пример новой ошибки преобразования: loan_amount было, но числом не стало
print('Примеры ошибок формата суммы:')
print(quarantine.loc[
    quarantine['reason'].str.contains('некорректная сумма кредита', na=False),
    ['source_row', 'loan_amount', 'income', 'reason']
].head())

# Пример необычно большой числовой суммы: правило её не ловит, нужен смысловой просмотр
print('Примеры необычно больших сумм:')
print(clean.loc[
    clean['loan_amount'] > 5_000_000,
    ['source_row', 'loan_amount', 'income']
].head())

Проверка года на испорченных данных: False
export сохранён
Примеры известных пропусков дохода:
    source_row  loan_amount  income
73          99     205000.0     NaN
76         103    3005000.0     NaN
77         104    1895000.0     NaN
79         106    2875000.0     NaN
80         107    2195000.0     NaN
Примеры ошибок формата дохода:
  source_row loan_amount income              reason
2          5    275000.0    BAD  некорректный доход
Примеры ошибок формата суммы:
  source_row loan_amount income                      reason
0          2        oops   72.0  некорректная сумма кредита
Примеры необычно больших сумм:
      source_row  loan_amount  income
85           112    6865000.0     NaN
249          326   57005000.0     NaN
1193        1531    6305000.0     NaN
1387        1781   40005000.0     NaN
1388        1782   15005000.0     NaN


## H+1. Конфликт исходных значений

## H+2. Совпадение опубликованных полей